In [ ]:
# SPDX-License-Identifier: MIT
# Copyright 2025-2026 Max Planck Institute for Security and Privacy (MPI-SP), University of Luebeck Institute for IT-Security (ITS)
import csv
import logging
import os
from pathlib import Path
import platform
import psutil
import subprocess
import sys
import time
from typing import Sequence, Tuple, Type
from multiprocessing import cpu_count

from sage.all import GF
from sage.rings.integer_ring import ZZ
from tqdm import tqdm

from ip_masking_gadgets import IPAddGadget, IPMultGadget, IPRefreshGadget, SecIPRefreshGadget
from observations import ObservationTupleManager
from polymasking_gadgets import Gadget, BgwMultGadget, LaolaMultGadget, OptRefresh, FOWZ25Refresh, RefreshSFRES18, SWComp, SWPolyAddGadget, SWPolyMulGadget, SWPolySubGadget
from security import SecurityNotion

logging.basicConfig(level=logging.INFO)

In [ ]:
# adjust this to your needs

# number of processes to use for verification
MAX_N_THREADS = cpu_count()
# output directory for the benchmarks
BENCHMARK_DIR = Path("benchmarks")

TIMEOUT = 15 * 60 # 15 minutes verification timeout

MAX_MEMORY = 1024 * 128

BENCHMARK_CSV_PATH = './../benchmarks/gadgets_to_benchmark.csv'


# heuristic for number of threads based on the number of observation tuples and an upper bound
def threading_heuristic(num_obstuples: int) -> int:
    if num_obstuples < 1000:
        return 1
    elif num_obstuples < 1000000:
        return min(32, MAX_N_THREADS)
    elif num_obstuples < 10000000:
        return min(64, MAX_N_THREADS)
    else:
        return MAX_N_THREADS

In [ ]:
# add all the Gadget classes here that you want to benchmark

GADGETS = {
    'SWPolyAddGadget': SWPolyAddGadget,
    'SWPolySubGadget': SWPolySubGadget,
    'SWPolyMulGadget': SWPolyMulGadget,
    'SWComp': SWComp,
    'BgwMultGadget': BgwMultGadget,
    'LaolaMultGadget': LaolaMultGadget,
    
    'Refresh': FOWZ25Refresh,
    'OptRefresh': OptRefresh,
    'RefreshSFRES18': RefreshSFRES18,
    'FOWZ25Refresh': FOWZ25Refresh, # not implemented yet?!

    'IPAddGadget': IPAddGadget,
    'IPRefreshGadget': IPRefreshGadget,
    'SecIPRefreshGadget': SecIPRefreshGadget,
    'IPMultGadget': IPMultGadget,
}

def expected_verification_result(sec_notion, t, gadget):
    # all gadgets should fulfill NI and t-probing, while only some fulfill SNI properties
    EXPECTED_RESULT = {
        # Polymasking gadgets
        SWPolyAddGadget: False if sec_notion is SecurityNotion.SNI else True,
        SWPolySubGadget: False if sec_notion is SecurityNotion.SNI else True,
        SWPolyMulGadget: False if sec_notion is SecurityNotion.SNI else True,
        SWComp: True,
        BgwMultGadget: True,
        LaolaMultGadget: True,

        # Refresh gadgets
        FOWZ25Refresh: True,
        OptRefresh: True,
        RefreshSFRES18: False if sec_notion is SecurityNotion.SNI and t > 2 else True, # This gadget is not SNI for t>2, but secure otherwise

        # IP gadgets
        IPAddGadget: False if sec_notion is SecurityNotion.SNI else True,
        IPRefreshGadget: False if sec_notion is SecurityNotion.SNI and t > 2 else True, # This gadget is not SNI for t>2, but secure otherwise
        SecIPRefreshGadget: True,
        IPMultGadget: True,
    }

    return EXPECTED_RESULT[type(gadget)]


# # add all the fields you want to benchmark
FIELDS = {
    'GF(2^8)': GF(2**8),
    'GF(7)': GF(7),
    'GF(3329)': GF(3329), # used in ML-KEM (Kyber)
    'ZZ': ZZ
}

# (G, F, snotion, (d, e, k, t), (expect_verification_success, expect_timeout))
TEST_CASES = []

def parse_gadgets_to_benchmark(file_path):
    gadgets = []
    with open(file_path, 'r', encoding='utf-8') as f:
        reader = csv.DictReader(f, delimiter=';')
        for row in reader:
            gadgets.append(row)
    return gadgets

benchmark_gadgets = parse_gadgets_to_benchmark(BENCHMARK_CSV_PATH)

for benchmark in tqdm(benchmark_gadgets):
    if benchmark['verify_security'].lower() == "true":
        try:
            gadget_type = GADGETS[benchmark['gadget']]
        except KeyError as exc:
            logging.warning(f"Unknown gadget {benchmark['gadget']}. Skipping.")
            continue

        try:
            snotion = SecurityNotion[benchmark['notion']]
        except KeyError as exc:
            logging.warning(f"Unknown security notion {benchmark['notion']}. Skipping.")
            continue

        try:
            field = FIELDS[benchmark['field']]
        except KeyError as exc:
            logging.warning(f"Unknown field {benchmark['field']}. Skipping.")
            continue

        try:
            d = int(benchmark['d'])
            e = int(benchmark['e'])
            k = int(benchmark['k'])
            t = int(benchmark['t'])

            if d <= 0 or k <= 0 or t <= 0:
                raise ValueError("Parameters d, k, t must be positive integers.")
            if e < 0:
                raise ValueError("Parameter e must be a non-negative integer.")
        except ValueError as exc:
            logging.warning(f"Invalid parameters d={benchmark['d']}, e={benchmark['e']}, k={benchmark['k']}, t={benchmark['t']} for gadget {benchmark['gadget']}, message: {exc}. Skipping.")
            continue

        try:
            expect_verif_success = True if benchmark['expected_security_result'].lower() == "true" else False
            expect_timeout = True if benchmark['timeout_expected'].lower() == "true" else False
        except Exception as exc:
            logging.warning(f"Invalid expected security result or timeout for gadget {benchmark['gadget']}: {exc}. Skipping.")
            continue

        TEST_CASES.append((gadget_type, field, snotion, (d, e, k, t), (expect_verif_success, expect_timeout)))

print(f"Parsed {len(TEST_CASES)} test cases from {BENCHMARK_CSV_PATH}.")

In [ ]:
# TODO: This is the copy-pasted (partially simplified) version of `verify_security()` from `ever.py`, remove duplicate code later on when there is an interface to obtain the actual number of observations.
def count_observations_in_obsgraph(
    security_notion: SecurityNotion,
    security_order: int,
    g: Gadget,
) -> int:
    obsgraph = g.obsgraph()
    svars = g.secret_vars_symbolic() # _encoded for PS
    pvars = g.public_vars()
    paravars = g.parameter_vars()

    output_observations = set()
    # ONLY For SNI
    if security_notion == SecurityNotion.SNI:
        output_observations = set(i.ni_obs.get_id() for i in g.output().sharing)
    
    # 0) lots of well-formedness checking
    all_secrets = set()
    for sv in svars:
        ssv = set(sv)
        all_secrets.update(ssv)

    support_vars_set = set(paravars)

    # 1) Filter out purely public observations (no secrets or randomness)
    tplMgr = ObservationTupleManager(obsgraph, security_notion, security_order, svars, output_observations)
    _ = tplMgr.filter_public_observations(pvars | support_vars_set, log=False)
    # 1b) Remove duplicate observations to reduce count of tuples
    tplMgr.filter_duplicate_observations(log=False)

    # Build tuples based on security notion
    num_tuples, _ = tplMgr.obs_tuple_iter()
    return num_tuples

In [ ]:
# ugly way to dynamically keep track of benchmarks to skip (due to timeout)
# (G, F, snotion, (d, e, k, t))
SKIP_BENCHMARKS = set()

def benchmark(G: Type, k: int, d: int, e: int, t: int, snotion: SecurityNotion, F: str, expected_verification_result: bool, expected_timeout: bool):
    global SKIP_BENCHMARKS
    assert issubclass(G, Gadget), f"G is not a gadget type!"
    print(f"Verifying {G.__name__}(k={k}, d={d}, e={e}, t={t}) in {F}.")

    if (G, F, snotion, (d, e, k, t)) in SKIP_BENCHMARKS:
        print(f"Skipping benchmark for {G.__name__}(k={k}, d={d}, e={e}, t={t}) in {F} due to previous timeout.")

        return {
            'num_tuples': None,
            'verification_threads': None,
            'verification_time': None,
            'verification_timeout': None,
            'verification_result': None,
            'verification_comment': "Previous timeout"
        }
    
    start = time.time_ns()

    try:
        gadget = G.from_file(d=d, k=k, e=e, field=F)
    except Exception as exc:
        logging.warning(f"Exception while trying to load gadget: {exc}")
        return {
            'num_tuples': None,
            'verification_threads': None,
            'verification_time': None,
            'verification_timeout': None,
            'verification_result': None,
            'verification_comment': "Gadget not found"
        }
    
    # Compute number of observation tuples to actually check
    obstuples = count_observations_in_obsgraph(snotion, t, gadget)
    num_threads = threading_heuristic(obstuples)

    comment = ""
    expected_res = None
    secure = None
    timeout = False

    start = time.time_ns()
    try:
        if snotion == SecurityNotion.NI:
            secure = gadget.verify_t_NI(t=t, num_worker=num_threads, log=False, continue_on_fail=False, timeout=TIMEOUT, max_memory_mb=MAX_MEMORY)
        elif snotion == SecurityNotion.SNI:
            secure = gadget.verify_t_SNI(t=t, num_worker=num_threads, log=False, continue_on_fail=False, timeout=TIMEOUT, max_memory_mb=MAX_MEMORY)
        elif snotion == SecurityNotion.PS:
            secure = gadget.verify_t_PS(t=t, num_worker=num_threads, log=False, continue_on_fail=False, timeout=TIMEOUT, max_memory_mb=MAX_MEMORY)
        else:
            raise ValueError(f"Unknown security notion {snotion} for gadget {G.__name__}.")

        # temporary workaround for not being able to catch TimeoutError from multithreaded verification instantiations
        if not secure and (time.time_ns() - start) / 1e9 > TIMEOUT:
            raise TimeoutError(f"Verification timeout")

        if expected_verification_result != secure:
            comment = f"Expected result [{expected_verification_result}] differs from actual result [{secure}]"

        if expected_timeout:
            comment = "Expected timeout, but verification went through within the timeout period."
    except TimeoutError as exc:
        print(f"Timeout for {G.__name__}(k={k}, d={d}, e={e}, t={t}) in {F} with security notion {snotion.name}. Verification time: {(time.time_ns() - start) / 1e9:.2f} seconds.")
        timeout = True
        if not expected_timeout:
            comment = "Timeout was not expected"
        # If we had a timeout, we skip benchmarks with higher values of e
        if (G, F, snotion, (d, e, k, t)) in TEST_CASES:
            for i in range(1, 10): # overapproximation
                print("WILL SKIP BENCHMARK for", (G, k, d, e+i, snotion, F))
                SKIP_BENCHMARKS.update(((G, F, snotion, (d, e+i, k, t))) for t in range(0,10))
            # special case: e=0 already fails, then we can skip all higher d values for arbitrary e as well
            if e == 0:
                for i in range(1,10):
                    for j in range(10): # overapproximation
                        print("WILL SKIP BENCHMARK for", (G, k, d+i, j, snotion, F))
                        SKIP_BENCHMARKS.update(((G, F, snotion, (d+i, j, k, t))) for t in range(0,10))
    except Exception as exc:
        print(f"Another exception occured during verification of {G.__name__}(k={k}, d={d}, e={e}, t={t}) in {F}, security notion {snotion.name}.: {exc}")
        secure = False
        comment = str(exc)
    finally:
        verification_time = (time.time_ns() - start) / 1e9

    return {
        'num_tuples': obstuples,
        'verification_threads': num_threads,
        'verification_time': verification_time,
        'verification_timeout': timeout,
        'verification_result': secure,
        'verification_comment': comment
    }

In [ ]:
def generate_benchmarks() -> Sequence[Tuple[Type, Type, SecurityNotion, int, int, int, int, bool, bool]]:
    return [(g, F, snotion, d, e, k, t, expect_verif_success, expect_timeout) for (g, F, snotion, (d, e, k, t), (expect_verif_success, expect_timeout)) in TEST_CASES]


def run_benchmarks(outfile: Path, benchmarks: Sequence[Tuple[Type, str, SecurityNotion, int, int, int, int, bool, bool]], orig_csv):
    COLUMNS = ["gadget", "k", "d", "e", "field", "verify_security", "t", "notion", "expected_security_result", "timeout_expected", "comment", "distinct_d_e", "num_tuples", "verification_threads", "verification_time", "verification_timeout", "verification_result", "verification_comment"]

    with open(outfile, 'w') as f:
        writer = csv.DictWriter(f, fieldnames=COLUMNS)
        writer.writeheader()

        pbar = tqdm(benchmarks)


        for (gadget, F, snotion, d, e, k, t, expect_verif_success, expect_timeout) in pbar:
            pbar.set_description(f"Now verifying {gadget.__name__}(k={k}, d={d}, e={e}) for t={t}")
            find_record = lambda rec: rec['gadget'] == gadget.__name__ and int(rec['k']) == k and int(rec['d']) == d and int(rec['e']) == e and int(rec['t']) == t and rec['notion'] == snotion.name and FIELDS[rec['field']] == F

            verification_results = benchmark(G=gadget, k=k, d=d, e=e, t=t, snotion=snotion, F=F, expected_verification_result=expect_verif_success, expected_timeout=expect_timeout)
            entry = list(filter(find_record, orig_csv))
            assert len(entry) == 1, f"Found none or multiple entries for given parameters {gadget.__name__}, {k}, {d}, {e}, {t}, {snotion.name}"
            entry = entry[0]
            entry['num_tuples'] = verification_results['num_tuples']
            entry['verification_threads'] = verification_results['verification_threads']
            entry['verification_time'] = verification_results['verification_time']
            entry['verification_timeout'] = verification_results['verification_timeout']
            entry['verification_result'] = verification_results['verification_result']
            entry['verification_comment'] = verification_results['verification_comment']

            writer.writerow(entry)
            f.flush()

In [ ]:
def entry_to_str(entry: Tuple[Type, Type, SecurityNotion, int, int, int, int, bool, bool]) -> str:
    gadget, F, snotion, d, e, k, t, _, _ = entry
    return f"{gadget.__name__}(k={k}, d={d}, e={e}, t={t} field={F}, notion={snotion.name})\n"

def write_config(cfg_name: Path, timestamp: str, benchmarks: Sequence[Tuple[Type, Tuple[str, Type], SecurityNotion, int, int, int]]):
    """Write the benchmark configuration to a file for later reference."""

    # obtain system information
    cpu = subprocess.run(["lscpu"], capture_output=True).stdout.decode('utf-8').splitlines()
    model_name = ', '.join(map(lambda l: l.split(':')[1].strip(), filter(lambda l: l.startswith('Model name'), cpu)))

    instances: Sequence[str] = []
    for (ctr, instance) in enumerate(benchmarks):
        instances.append(f"{ctr:<3}: {entry_to_str(instance)}")


    with open(cfg_name, 'w') as cfg_file:
        cfg_file.write("# eVer Benchmark Configuration\n")
        cfg_file.write("# System Information\n")
        cfg_file.write(f"timestamp        = {timestamp}\n")
        cfg_file.write(f"machine_name     = {platform.node()}\n")
        cfg_file.write(f"cpu_model        = {model_name}\n")
        cfg_file.write(f"cpu_count        = {os.cpu_count()}\n")
        cfg_file.write(f"memory           = {psutil.virtual_memory().total // (1024**3)} GB\n")
        cfg_file.write(f"python_version   = {platform.python_version()}\n")
        cfg_file.write("# Benchmark parameters\n")
        cfg_file.write(f"max_threads      = {MAX_N_THREADS}\n")
        cfg_file.write(f"gadget_list      = {', '.join([x[0].__name__ for x in benchmarks])}\n")
        cfg_file.write(f"num_instances    = {len(benchmarks)}\n")
        cfg_file.write("#\n")
        cfg_file.write("# Benchmark results for the following instances will be generated and stored in the CSV file.\n")
        cfg_file.write("# Each line corresponds to a benchmark instance.\n")
        cfg_file.write("# Format: gadget(k, d, e, field, notion)\n")
        cfg_file.write("# Instances:\n")
        cfg_file.writelines(instances)


def main():
    timestamp = time.strftime("%Y%m%d-%H%M%S")
    bench_path = BENCHMARK_DIR / timestamp
    print("Bench path:", bench_path)
    sys.stdout.flush()
    os.makedirs(bench_path, exist_ok=True)

    benchmark_configs: Sequence[Sequence[Tuple[Type, Type, SecurityNotion, int, int, int]]] = generate_benchmarks()

    write_config(bench_path / "ever_benchmark.conf", timestamp, benchmark_configs)
    sys.stdout = open(bench_path / "ever_benchmark.log", "w")
    run_benchmarks(bench_path / "ever_benchmark.csv", benchmark_configs, benchmark_gadgets)

main()